# Week 4 — Final Findings & Reporting
**Project:** Food Delivery Data Analysis  |  **Notebook:** `04_final_analysis.ipynb`

**Goal:** (1) verify the work from Weeks 1–3, (2) state 7 evidence-based key findings, (3) propose three operational ideas to investigate, and (4) conclude with limitations and next steps.

**How to read this notebook:** every number below is recalculated here from `data/processed/cleaned_food_delivery.csv`. Findings describe what the data shows; interpretations are cautious and, because this is observational data, **associations are never presented as proof of cause and effect**. Recommendations are ideas to investigate, not guaranteed improvements.

## 1. Setup and data loading
**What / why:** Load the cleaned dataset with the documented path and standardise column names, exactly as in Weeks 2 and 3. If the cleaned file has no `distance_km`, it is derived from coordinates (straight-line haversine, placeholder coordinates excluded).

In [1]:
import os, json, glob
import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
DATA_PATH = "../data/processed/cleaned_food_delivery.csv"
df = pd.read_csv(DATA_PATH)
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
TARGET = "time_taken_min"

if "distance_km" not in df.columns:
    c = ["restaurant_latitude", "restaurant_longitude", "delivery_location_latitude", "delivery_location_longitude"]
    ok = (df[c] > 0.1).all(axis=1)
    lat1, lon1, lat2, lon2 = [np.radians(df[x].abs()) for x in c]
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    df["distance_km"] = np.where(ok, 6371 * 2 * np.arcsin(np.sqrt(a)), np.nan)

N = len(df); N_DIST = int(df["distance_km"].notna().sum())
print(f"Loaded {DATA_PATH}: {N:,} rows x {df.shape[1]} columns | valid distance rows: {N_DIST:,}")

Loaded ../data/processed/cleaned_food_delivery.csv: 45,584 rows x 21 columns | valid distance rows: 41,944


## 2. Project verification
**What / why:** The brief asks me to verify the complete project before concluding. I check four things: (a) required files exist, (b) the earlier notebooks contain executed results with no error outputs, (c) the cleaned data is internally sound, and (d) the key numbers reported in Weeks 2 and 3 can be reproduced from the cleaned dataset.

### 2.1 Required files and notebook audit

In [2]:
required = ["../data/processed/cleaned_food_delivery.csv", "../docs/cleaning_log.csv", "../README.md", "../requirements.txt",
            "../data/raw/README.md",
            "../visualizations/01_delivery_duration_distribution.png", "../visualizations/02_traffic_comparison.png",
            "../visualizations/03_weather_comparison.png", "../visualizations/04_distance_vs_duration.png",
            "../visualizations/05_vehicle_comparison.png"]
files = pd.DataFrame({"file": [p.replace("../", "") for p in required],
                      "exists": [os.path.exists(p) for p in required],
                      "size_kb": [round(os.path.getsize(p) / 1024, 1) if os.path.exists(p) else None for p in required]})
display(files)

# Audit the earlier notebooks: executed code cells and any error outputs saved in them
audit = []
for name in ["01_data_understanding_and_cleaning", "02_exploratory_data_analysis", "03_visualization"]:
    p = f"../notebooks/{name}.ipynb"
    if not os.path.exists(p):
        audit.append({"notebook": name, "found": False, "code_cells": None, "not_executed": None, "error_outputs": None}); continue
    cells = [c for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
    audit.append({"notebook": name, "found": True, "code_cells": len(cells),
                  "not_executed": sum(1 for c in cells if c.get("execution_count") is None and c["source"]),
                  "error_outputs": sum(1 for c in cells for o in c.get("outputs", []) if o.get("output_type") == "error")})
audit = pd.DataFrame(audit)
display(audit)

,file,exists,size_kb
0,data/processed/cleaned_food_delivery.csv,True,"6,545.60"
1,docs/cleaning_log.csv,True,3.10
2,README.md,True,20.20
3,requirements.txt,True,0.10
4,data/raw/README.md,True,3.50
5,visualizations/01_delivery_duration_distributi...,True,57.10
6,visualizations/02_traffic_comparison.png,True,56.40
7,visualizations/03_weather_comparison.png,True,77.70
8,visualizations/04_distance_vs_duration.png,True,352.40
9,visualizations/05_vehicle_comparison.png,True,63.70


,notebook,found,code_cells,not_executed,error_outputs
0,01_data_understanding_and_cleaning,True,18,0,0
1,02_exploratory_data_analysis,True,14,0,0
2,03_visualization,True,7,0,0


**Result:** 10 of 10 required files were found in this environment. The notebook audit found no error outputs in any notebook that was present. Notebook(s) not found here: 01_data_understanding_and_cleaning — re-run this section from inside the full repository so they are included in the audit. 

### 2.2 Data integrity checks

In [3]:
t = df[TARGET]
integrity = pd.DataFrame({"check": ["Rows", "Duplicate rows", "Duplicate order IDs", "Missing delivery duration",
                                    "Delivery duration <= 0", "Min / max delivery duration (min)",
                                    "Missing traffic label", "Missing weather label", "Missing vehicle type",
                                    "Rows without valid distance"],
                          "value": [f"{N:,}", int(df.duplicated().sum()),
                                    int(df["id"].duplicated().sum()) if "id" in df.columns else "n/a (no id column)",
                                    int(t.isna().sum()), int((t <= 0).sum()), f"{t.min()} / {t.max()}",
                                    int(df["road_traffic_density"].isna().sum()), int(df["weather_conditions"].isna().sum()),
                                    int(df["type_of_vehicle"].isna().sum()), N - N_DIST]})
display(integrity)

,check,value
0,Rows,"45,584"
1,Duplicate rows,0
2,Duplicate order IDs,0
3,Missing delivery duration,0
4,Delivery duration <= 0,0
5,Min / max delivery duration (min),10 / 54
6,Missing traffic label,601
7,Missing weather label,616
8,Missing vehicle type,0
9,Rows without valid distance,3640


**Result:** The cleaned file has 45,584 rows, 0 duplicate rows and 0 duplicate order IDs. Delivery duration has no missing or non-positive values and ranges from 10 to 54 minutes. 601 rows lack a traffic label, 616 lack a weather label, and 3640 rows have no valid distance (placeholder coordinates). These rows are excluded only from the specific comparison that needs them, never imputed.

### 2.3 Reproducing the numbers reported in Weeks 2 and 3
**What / why:** The "reported" column holds values copied from the Week 2 and Week 3 notebooks; the "recalculated" column is computed here from the cleaned data. A mismatch would mean a written statement is wrong or the data changed.

In [4]:
def _med(col, k): return df.groupby(col)[TARGET].median()[k]
def _mean(col, k): return df.groupby(col)[TARGET].mean()[k]
dd = df.dropna(subset=["distance_km", TARGET])
slope_chk = np.polyfit(dd["distance_km"], dd[TARGET], 1)[0]
rt = df.dropna(subset=["delivery_person_ratings"]); ag = df.dropna(subset=["delivery_person_age"])

checks = [  # (item, reported value in Week 2/3, recalculated here)
    ("Rows in cleaned data", 45584, N),
    ("Mean delivery time (min)", 26.29, t.mean()), ("Median delivery time (min)", 26, t.median()),
    ("Q1 (min)", 19, t.quantile(.25)), ("Q3 (min)", 32, t.quantile(.75)), ("Std deviation (min)", 9.38, t.std()),
    ("Median, Low traffic", 20, _med("road_traffic_density", "Low")), ("Median, Medium traffic", 27, _med("road_traffic_density", "Medium")),
    ("Median, High traffic", 27, _med("road_traffic_density", "High")), ("Median, Jam traffic", 31, _med("road_traffic_density", "Jam")),
    ("Mean, Cloudy weather", 28.92, _mean("weather_conditions", "Cloudy")), ("Mean, Fog weather", 28.91, _mean("weather_conditions", "Fog")),
    ("Mean, Sunny weather", 21.86, _mean("weather_conditions", "Sunny")),
    ("Rows with valid distance", 41944, len(dd)), ("Pearson r (distance, duration)", 0.320, dd["distance_km"].corr(dd[TARGET])),
    ("Slope (min per km)", 0.54, slope_chk),
    ("Median, motorcycle", 26, _med("type_of_vehicle", "motorcycle")), ("Median, scooter", 24, _med("type_of_vehicle", "scooter")),
    ("Median, electric scooter", 24, _med("type_of_vehicle", "electric_scooter")),
    ("Deliveries, motorcycle", 26429, int((df.type_of_vehicle == "motorcycle").sum())), ("Deliveries, bicycle", 68, int((df.type_of_vehicle == "bicycle").sum())),
    ("Mean, 0 concurrent orders", 22.88, df[df.multiple_deliveries == 0][TARGET].mean()),
    ("Mean, 3 concurrent orders", 47.82, df[df.multiple_deliveries == 3][TARGET].mean()),
    ("r (partner age, duration)", 0.300, ag["delivery_person_age"].corr(ag[TARGET])),
    ("r (partner rating, duration)", -0.341, rt["delivery_person_ratings"].corr(rt[TARGET])),
]
verify = pd.DataFrame(checks, columns=["item", "reported", "recalculated"])
verify["difference"] = (verify["recalculated"] - verify["reported"]).abs()
verify["status"] = np.where(verify["difference"] <= 0.0051, "match", "CHECK")     # tolerance = rounding to 2 decimals
display(verify.style.format({"reported": "{:,.3f}", "recalculated": "{:,.3f}", "difference": "{:.4f}"}))
print(f"{(verify.status == 'match').sum()} of {len(verify)} reported numbers reproduced.")

,item,reported,recalculated,difference,status
0,Rows in cleaned data,"45,584.000","45,584.000",0.0000,match
1,Mean delivery time (min),26.290,26.294,0.0040,match
2,Median delivery time (min),26.000,26.000,0.0000,match
3,Q1 (min),19.000,19.000,0.0000,match
4,Q3 (min),32.000,32.000,0.0000,match
5,Std deviation (min),9.380,9.384,0.0043,match
6,"Median, Low traffic",20.000,20.000,0.0000,match
7,"Median, Medium traffic",27.000,27.000,0.0000,match
8,"Median, High traffic",27.000,27.000,0.0000,match
9,"Median, Jam traffic",31.000,31.000,0.0000,match


25 of 25 reported numbers reproduced.


**Result:** 25 of 25 numbers reported in Weeks 2 and 3 are reproduced from the cleaned dataset (tolerance: rounding to two decimals). There are no mismatches, so the earlier written statements are consistent with the data. All five charts are produced by `03_visualization.ipynb`, which reads the same `cleaned_food_delivery.csv`.

## 3. Calculating the evidence for the key findings
**What / why:** One cell recalculates every metric used in the findings so that each number in section 4 can be traced to code. **Expectation:** the same patterns as Weeks 2 and 3.

In [5]:
T = TARGET
# --- Finding 1: baseline
mean_v, median_v, sd_v = t.mean(), t.median(), t.std()
q1, q3 = t.quantile([.25, .75])
share_15_35 = ((t >= 15) & (t <= 35)).mean() * 100
n_45 = int((t >= 45).sum()); pct_45 = n_45 / N * 100

# --- Finding 2: traffic
order_tr = ["Low", "Medium", "High", "Jam"]
tr = df.dropna(subset=["road_traffic_density"]).groupby("road_traffic_density")[T].agg(count="count", median="median", mean="mean", std="std").reindex(order_tr)
tr_share_jam = tr.loc["Jam", "count"] / tr["count"].sum() * 100
tr_dist = df.groupby("road_traffic_density")["distance_km"].mean().reindex(order_tr)

# --- Finding 3: weather (and weather x traffic)
wt = df.dropna(subset=["weather_conditions"]).groupby("weather_conditions")[T].agg(count="count", mean="mean").sort_values("mean", ascending=False)
piv = df.pivot_table(index="road_traffic_density", columns="weather_conditions", values=T, aggfunc="median").reindex(order_tr)
sunny_rng = (piv["Sunny"].min(), piv["Sunny"].max())
cf_med_high = piv.loc[["Medium", "High", "Jam"], ["Cloudy", "Fog"]]

# --- Finding 4: concurrent orders
mu = df.dropna(subset=["multiple_deliveries"]).groupby("multiple_deliveries")[T].agg(count="count", mean="mean").sort_index()
mu_tr = df.pivot_table(index="road_traffic_density", columns="multiple_deliveries", values=T, aggfunc="mean").reindex(order_tr)
mu_share_2plus = mu.loc[[2.0, 3.0], "count"].sum() / mu["count"].sum() * 100

# --- Finding 5: distance
pearson = dd["distance_km"].corr(dd[T]); spearman = dd["distance_km"].corr(dd[T], method="spearman")
slope, intercept = np.polyfit(dd["distance_km"], dd[T], 1)
band = pd.cut(dd["distance_km"], [0, 5, 10, 15, 25], labels=["0-5", "5-10", "10-15", "15+"])
band_tbl = dd.groupby(band, observed=True)[T].agg(count="count", mean="mean")
n_gt15 = int((dd.distance_km > 15).sum()); long_fast = int(((dd.distance_km > 15) & (dd[T] <= 20)).sum())

# --- Finding 6: vehicle
vh = df.groupby("type_of_vehicle")[T].agg(count="count", median="median", mean="mean")
vh["mean_distance_km"] = df.groupby("type_of_vehicle")["distance_km"].mean()
vh["mean_concurrent"] = df.groupby("type_of_vehicle")["multiple_deliveries"].mean()

# --- Finding 7: partner proxies (age and rating are NOT direct experience measures)
r_age = ag["delivery_person_age"].corr(ag[T]); r_rating = rt["delivery_person_ratings"].corr(rt[T])
rg = pd.cut(rt["delivery_person_ratings"], [0, 4.0, 4.5, 4.8, 5.0], labels=["<=4.0", "4.1-4.5", "4.6-4.8", "4.9-5.0"])
rating_tbl = rt.groupby(rg, observed=True)[T].agg(count="count", mean="mean")
ag_grp = pd.cut(ag["delivery_person_age"], [14, 29, 60], labels=["under 30", "30+"])
age_tbl = ag.groupby(ag_grp, observed=True)[T].agg(count="count", mean="mean")

for name, obj in [("Traffic", tr), ("Weather", wt), ("Concurrent orders", mu), ("Vehicle", vh)]:
    print(name); display(obj)
print(f"Traffic x weather median (Sunny range {sunny_rng[0]:.0f}-{sunny_rng[1]:.0f}); Cloudy/Fog at Medium-Jam range {cf_med_high.values.min():.0f}-{cf_med_high.values.max():.0f}")

Traffic


,count,median,mean,std
road_traffic_density,,,,
Low,15476,20.00,21.27,6.80
Medium,10945,27.00,26.70,8.56
High,4423,27.00,27.24,8.40
Jam,14139,31.00,31.18,9.94


Weather


,count,mean
weather_conditions,,
Cloudy,7533,28.92
Fog,7653,28.91
Windy,7422,26.12
Sandstorms,7494,25.88
Stormy,7584,25.87
Sunny,7282,21.86


Concurrent orders


,count,mean
multiple_deliveries,,
0.00,14094,22.88
1.00,28151,26.85
2.00,1985,40.45
3.00,361,47.82


Vehicle


,count,median,mean,mean_distance_km,mean_concurrent
type_of_vehicle,,,,,
bicycle,68,26.00,26.43,9.44,0.76
electric_scooter,3814,24.00,24.47,9.75,0.70
motorcycle,26429,26.00,27.61,9.69,0.78
scooter,15273,24.00,24.48,9.76,0.69


Traffic x weather median (Sunny range 19-21); Cloudy/Fog at Medium-Jam range 28-37


## 4. Key findings
Each finding gives the observation, numerical evidence and a cautious operational interpretation. The chart images are the ones exported in Week 3.

### Finding 1 — Most deliveries fall in a predictable middle range, with a minority of long deliveries
- **Observation:** Delivery time is clustered around the middle of the range with a mild right tail.
- **Numerical evidence:** mean 26.29 min, median 26 min, SD 9.38; the middle half lies between 19 and 32 min; 74.3% of deliveries take 15–35 min; 1,803 deliveries (3.96%) take 45 min or more (maximum 54) (n = 45,584).
- **Operational interpretation:** A typical order arrives in about 26 minutes, so overall performance looks consistent; the long-delivery minority is where investigation is most useful.

![Delivery duration distribution](../visualizations/01_delivery_duration_distribution.png)

### Finding 2 — Heavier traffic is associated with slower and less predictable deliveries
- **Observation:** Median time rises from Low to Jam traffic, and the spread widens.
- **Numerical evidence:** median 20 min (Low, n=15,476), 27 (Medium, n=10,945), 27 (High, n=4,423), 31 (Jam, n=14,139); standard deviation 6.8 min in Low vs 9.9 in Jam. Jam accounts for 31.4% of orders with a traffic label.
- **Operational interpretation:** Jam conditions are common and are linked to both longer and more variable delivery times, which makes promised arrival times harder to meet. High traffic is not slower than Medium, but High orders are also shorter on average (4.4 km vs 11.1 km), so traffic and distance are entangled.

![Traffic comparison](../visualizations/02_traffic_comparison.png)

### Finding 3 — Cloudy and Fog weather coincide with slower deliveries, mainly when traffic is not light
- **Observation:** Average time differs by weather, and the gap depends on traffic level.
- **Numerical evidence:** mean 28.92 min for Cloudy (n=7,533) and 28.91 for Fog (n=7,653) vs 21.86 for Sunny (n=7,282), a gap of 7.1 min. Under Sunny weather the median stays at 19–21 min at every traffic level, whereas Cloudy/Fog with Medium-to-Jam traffic have medians of 28–37 min.
- **Operational interpretation:** Weather and traffic seem to act together rather than separately. Weather may also overlap with city or season, so this is an association and not an isolated weather effect.

![Weather comparison](../visualizations/03_weather_comparison.png)

### Finding 4 — Carrying several orders at once shows the steepest increase in delivery time
- **Observation:** Mean delivery time increases with the number of concurrent orders (`multiple_deliveries`).
- **Numerical evidence:** mean 22.88 min with 0 (n=14,094), 26.85 with 1 (n=28,151), 40.45 with 2 (n=1,985), 47.82 with 3 (n=361). The rise appears at every traffic level (Low traffic: 19.2 → 43.6 min; Jam: 27.5 → 48.2 min). Orders with 2 or 3 concurrent deliveries are only 5.3% of orders with this field.
- **Operational interpretation:** Batching orders is strongly associated with slower individual deliveries, even in light traffic. Because the 2 and 3 groups are small, this should be confirmed before it drives decisions.

### Finding 5 — Distance matters, but explains only a small share of delivery time
- **Observation:** Longer straight-line distance goes with longer time, with a weak-to-moderate relationship that flattens beyond about 10 km.
- **Numerical evidence:** Pearson r = 0.320, Spearman ρ = 0.319 (n = 41,944); about 0.54 min per km; r² = 0.103. Mean time is 22.2 min for 0–5 km and 29.8 / 29.9 min for 10–15 / 15+ km. 1,605 of 8,046 trips over 15 km (20%) took 20 min or less.
- **Operational interpretation:** Distance alone is a poor predictor of delivery time (about 10% of variation), so ETAs based only on distance would miss most of the variation. Distance is straight-line, not road distance.

![Distance vs duration](../visualizations/04_distance_vs_duration.png)

### Finding 6 — Motorcycles are slightly slower than scooters, and distance or load do not explain the gap
- **Observation:** Motorcycle deliveries have a higher median and mean than scooter and electric scooter deliveries.
- **Numerical evidence:** median 26 min (motorcycle, n=26,429) vs 24 (scooter, n=15,273) and 24 (electric scooter, n=3,814); means 27.61 vs 24.48 and 24.47 min. Mean distance is similar (9.69 vs 9.76 km) and so is average concurrent load (0.78 vs 0.69). Bicycle has only 68 deliveries.
- **Operational interpretation:** A gap of about 3.1 min in the mean exists that these two fields do not explain; the reason (areas served, partner mix, order mix) is unknown. The box plots overlap heavily, so vehicle type is a modest factor.

![Vehicle comparison](../visualizations/05_vehicle_comparison.png)

### Finding 7 — Partner age and rating are linked to delivery time, but they are not measures of experience
- **Observation:** Lower-rated and older partners have longer average delivery times.
- **Numerical evidence:** correlation with duration: rating r = -0.341 (n=43,623), age r = 0.300 (n=43,730). Partners rated ≤4.0 average 35.9 min (n=2,444) vs 24.3 min for 4.6–4.8 (n=21,224); partners aged 30+ average 29.5 min vs 23.0 for under 30.
- **Operational interpretation:** The dataset has no tenure field, so experience cannot be measured. Ratings may reflect slow deliveries (reverse direction), and age may stand in for other differences, so these patterns are leads for investigation, not conclusions about partner skill.

## 5. Operational recommendations (ideas to investigate)
These are hypotheses to test, based on the findings above. None is a guaranteed improvement, and each lists the validation needed before acting.

### Recommendation 1 — Review how many concurrent orders a partner carries
- **Issue / pattern observed:** Delivery time rises sharply with concurrent orders (Finding 4).
- **Numerical evidence:** mean 22.88 min with 0 concurrent orders vs 40.45 with 2 and 47.82 with 3; the increase holds at every traffic level (Jam: 27.5 → 48.2 min).
- **Proposed action:** Trial a limit or a dispatch rule that avoids assigning 2–3 concurrent orders, at least in Medium-to-Jam traffic, and compare delivery times against a control group.
- **Operational area that may improve:** Dispatch and batching rules; delivery-time consistency.
- **Further validation needed:** Only 2,346 orders have 2–3 concurrent deliveries, so confirm with more data; check whether those orders involve longer routes or specific partners; weigh the time cost against cost-per-order savings from batching.

### Recommendation 2 — Make arrival-time estimates sensitive to traffic and weather together
- **Issue / pattern observed:** Delay is concentrated in specific combinations, and distance alone predicts little (Findings 2, 3, 5).
- **Numerical evidence:** medians are 19–21 min in Sunny weather at every traffic level, but 28–37 min for Cloudy/Fog with Medium-to-Jam traffic; distance explains about 10% of variation (r = 0.320).
- **Proposed action:** Test condition-based ETA buffers (traffic × weather) in place of a distance-only estimate, and flag Cloudy/Fog + heavy-traffic periods for extra planning.
- **Operational area that may improve:** Accuracy of promised delivery times and customer expectations.
- **Further validation needed:** Compare estimated vs actual times (not in this dataset); add time-of-day and city/season analysis to see whether weather simply proxies for place or time; use road distance rather than straight-line distance.

### Recommendation 3 — Investigate the motorcycle–scooter gap and slower, lower-rated partner groups
- **Issue / pattern observed:** Motorcycle deliveries are slower than scooter deliveries without a difference in distance or load, and lower-rated partners are slower (Findings 6 and 7).
- **Numerical evidence:** mean 27.61 min (motorcycle) vs 24.48 (scooter) with mean distance 9.69 vs 9.76 km; partners rated ≤4.0 average 35.9 min vs 24.3 min for 4.6–4.8.
- **Proposed action:** Run a focused review of the areas, order types and partners behind the slower groups (for example by comparing like-for-like trips) before changing vehicle assignment or partner policy.
- **Operational area that may improve:** Fleet allocation and partner support.
- **Further validation needed:** The dataset has no partner tenure, area or order-preparation time, so collect those fields; ratings may be the result of slow deliveries, so do not treat them as a cause; avoid conclusions for bicycle (n = 68).

## 6. Conclusion
**What was analysed:** 45,584 food-delivery orders, to understand how delivery time (mean 26.29 min, median 26 min) varies with traffic, weather, distance, vehicle type, concurrent orders and partner-related fields.

**What was found:** Most deliveries are completed within a fairly narrow range (74.3% in 15–35 min). The largest observed differences are linked to the number of concurrent orders (mean 22.9 → 47.8 min), heavy traffic (median 20 → 31 min) and Cloudy/Fog weather (about 7 min above Sunny). Lower partner ratings are also linked to slower deliveries (35.9 min for ratings ≤4.0 vs 24.3 min for 4.6–4.8), although rating is only a proxy and may reflect slow deliveries. Distance (r = 0.32) and vehicle type are smaller factors.

**Limitations and assumptions:**
- All results are associations from observational data; traffic, weather, distance, load, vehicle and partner fields overlap, so no cause-and-effect claim is made.
- Distance is straight-line (haversine) from coordinates, and 3,640 rows (8.0%) with placeholder coordinates have no distance.
- Rows with a missing traffic (601) or weather (616) label were excluded from those comparisons, not imputed.
- There is no partner tenure field; age and rating are proxies only. Small groups (bicycle n = 68, 2–3 concurrent orders, Semi-Urban city, festival days) limit some comparisons.
- Delivery times stop at 54 minutes, so very long deliveries are not represented.

**Possible next steps:** collect time-of-day, area, order-preparation time and partner tenure; use road distance; test the concurrent-order and ETA ideas in a controlled trial; and compare estimated with actual delivery times.